# CyberAccountant pipeline — interactive driver

Run cells top to bottom. Each section runs one stage and shows its artifact.

Config: change `ID_STYLE` between `"mnemonic"` and `"numeric"` to compare.

In [1]:
from pathlib import Path
from checkier import Pipeline, PipelineConfig

ID_STYLE = "mnemonic"          # or "numeric"
TAG_MODEL = "deepseek/deepseek-v4.1-flash-20261017"
# NB: the old "~deepseek/deepseek-pro-latest" id 404s on this account —
# relations run fine on the flash model too.
REL_MODEL = TAG_MODEL

cfg = PipelineConfig(
    data_dir=Path("../data_md"),
    out_dir=Path("../out"),
    id_style=ID_STYLE,
    tag_model=TAG_MODEL,
    rel_model=REL_MODEL,
)
from checkier.config import load_api_credentials
from infrastructure.llm import HttpModelAdapter
from processing.common.llm import ModelClient
import logging
logging.basicConfig(level=logging.INFO)
p = Pipeline(cfg, client=ModelClient(HttpModelAdapter(*load_api_credentials())), logger=logging.getLogger("pipeline"))

## Stage 0 — ingest / rename / dedup

In [2]:
mapping = p.ingest()
import pandas as pd
pd.DataFrame(mapping).T

,original,role,sha256,duplicate_of,tag
0f1a7407,данные/02_01_Ведомость_ЗП_Январь_2026.xlsx.md,data,4aa8154585ee1c226d3434b25fed004d2463a6e5e5c195...,NaN,NaN
d865f279,данные/02_02_Ведомость_ЗП_Февраль_2026.xlsx.md,data,c4ccaff6c874f63c26af97d4de448f9a0dc3ac1f811349...,NaN,NaN
40aac61f,данные/02_03_Ведомость_ЗП_Март_2026.xlsx.md,data,5e25554b83e1c434ef4c6befdc0b43ebba05f8ce83bdd9...,NaN,NaN
e35d4a06,данные/02_04_Ведомость_ЗП_Апрель_2026.xlsx.md,data,4e108b7cc42f3d987d779e1c255bc3403ed969cde1c52e...,NaN,NaN
85577f42,данные/02_05_Ведомость_ЗП_Май_2026.xlsx.md,data,823f89a747de7db43d6a8c25ee98cb00b92bea18879527...,NaN,NaN
57c8d0b9,данные/03_01_Страховые_взносы_Январь_2026.xlsx.md,data,f577cc50ec3cebb0f47b7a5f14f511871dd4e3723af3a8...,NaN,NaN
dbb4fc37,данные/03_02_Страховые_взносы_Февраль_2026.xls...,data,77d1da207f88b9eef815aabfb42b2e7c6f5362403d1007...,NaN,NaN
4ae2605a,данные/03_03_Страховые_взносы_Март_2026.xlsx.md,data,cb481eef1a047a8cecf7d255b900efbe9c73a4bf48181a...,NaN,NaN
9f63aa1b,данные/03_04_Страховые_взносы_Апрель_2026.xlsx.md,data,b988a7139882f844ed169c3f7dc0c5c64896f56bd30411...,NaN,NaN
ac5e6591,данные/03_05_Страховые_взносы_Май_2026.xlsx.md,data,8414801ec16918fd770ae8505a267e967480b80c13d94c...,NaN,NaN


## Stage 1 — entity tagging (LLM, per file)

In [3]:
from checkier.stage1_tag import TagProgress, TagStats


def on_tag_progress(prog: TagProgress, stats: TagStats) -> None:
    """Swappable live-progress hook (forward to a frontend here later)."""
    if prog.ok:
        line = (
            f"[{stats.done}/{stats.total}] OK  {prog.file}  "
            f"entities={prog.entities} retries={prog.retries}"
        )
    else:
        line = f"[{stats.done}/{stats.total}] ERR {prog.file}  {prog.error}"
    print(line, flush=True)


tagged = p.tag(on_progress=on_tag_progress)
print(f"\nDone: {len(tagged)} files tagged.")
list(tagged.keys())[:5], len(tagged)

[1/24] OK  e35d4a06  entities=9 retries=0
[2/24] OK  d865f279  entities=8 retries=0
[3/24] OK  85577f42  entities=8 retries=0
[4/24] OK  0f1a7407  entities=10 retries=0
[5/24] OK  4e7519a8  entities=6 retries=0
[6/24] OK  9f63aa1b  entities=6 retries=0
[7/24] OK  94523f86  entities=5 retries=0
[8/24] OK  40aac61f  entities=8 retries=1
[9/24] OK  57c8d0b9  entities=6 retries=2
[10/24] OK  ac5e6591  entities=6 retries=1
[11/24] OK  4ae2605a  entities=6 retries=2
[12/24] OK  92fdb06c  entities=6 retries=0
[13/24] OK  dbb4fc37  entities=6 retries=2
[14/24] OK  47d6940b  entities=4 retries=0
[15/24] OK  07213099  entities=5 retries=0
[16/24] OK  6d97b3ec  entities=4 retries=0
[17/24] OK  67a203c6  entities=4 retries=0
[18/24] OK  049cdd29  entities=4 retries=0
[19/24] OK  f3473dac  entities=5 retries=0
[20/24] OK  b4c8366e  entities=5 retries=0
[21/24] OK  e98f8dc2  entities=4 retries=0
[22/24] OK  012a8029  entities=5 retries=0
[23/24] OK  df27026d  entities=7 retries=0
[24/24] OK  0dff636

(['e35d4a06', 'd865f279', '85577f42', '0f1a7407', '4e7519a8'], 24)

## Stage 2 — global entity registry

In [4]:
reg = p.registry()
import pandas as pd
rows = [{"num": e.num, "mnemonic": e.mnemonic, "type": e.type,
         "raw": e.raw, "value": e.value, "unit": e.unit, "file": e.file,
         "role": e.role} for e in reg.entities]
df = pd.DataFrame(rows)
df.head(20)

/home/alice/code/shitposting/CyberAccountant/src/checkier/bip39.py:90: RuntimeWarning: BIP39 wordlist not found; using deterministic generated fallback. For the canonical list run: python scripts/fetch_bip39.py
  self._a, self._b = load_words()


,num,mnemonic,type,raw,value,unit,file,role
0,0,dere-gidy,number,5,5,,012a8029,request
1,1,cusa-zypu,number,2026,2026,,012a8029,request
2,2,vybi-jiku,contract,24/03/15-01,24/03/15-01,,012a8029,request
3,3,nyce-jamo,date,01.01.2026,739617,,012a8029,request
4,4,hela-gaqu,date,31.05.2026,739767,,012a8029,request
5,5,vexa-refo,docid,УПР-55,УПР-55,,049cdd29,data
6,6,mala-siry,date,29.05.2026,739765,,049cdd29,data
7,7,qeba-faqu,contract,24/03/15-01,24/03/15-01,,049cdd29,data
8,8,rade-gydo,amount,"744 000,00",744000.00,RUB,049cdd29,data
9,9,fubi-basy,docid,0426-77,0426-77,,07213099,data


In [5]:
df["type"].value_counts()

type
amount      85
number      60
date        33
contract    25
docid       24
Name: count, dtype: int64

## Stage 2b — date normalization (LLM, one batch call)

The deterministic parser handles `DD.MM.YYYY` and `DD month YYYY`, but not
period labels like `Январь 2026` — those entities get `value = None` and
would stay unknowns in the solver. This stage asks the model for concrete
dates (`Январь 2026` → `01.01.2026`, first day of the period). Provenance
is kept in `Entity.normalized_from`.

In [6]:
changed = p.normalize_dates()
print(f"normalized {len(changed)} date entities")
[(e.mnemonic, e.normalized_from, e.value) for e in changed]

normalized 18 date entities


[('dehi-tysy', 'Август 2025', 739464),
 ('bixu-sabe', 'Май 2026', 739737),
 ('nyce-mebu', 'августа 2025', 739464),
 ('nezy-qewo', 'мая 2026', 739737),
 ('seqi-dyty', 'Январь_2026', 739617),
 ('zuro-novo', 'Январь 2026', 739617),
 ('vylo-mixa', 'Март 2026', 739676),
 ('musi-syfe', 'Март 2026', 739676),
 ('rari-lacu', 'январь 2026', 739617),
 ('viqu-gegy', 'Январь 2026', 739617),
 ('mofo-kapa', 'март 2026', 739676),
 ('gyti-xoke', 'Май 2026', 739737),
 ('bono-fudo', 'Апрель 2026', 739707),
 ('mofu-juco', 'Май 2026', 739737),
 ('tire-wysy', 'Февраль 2026', 739648),
 ('daga-nixe', 'Февраль 2026', 739648),
 ('lehi-govy', 'Апрель 2026', 739707),
 ('moso-dyhy', 'июнь 2026', 739768)]

## Stage 3 — replace tags with ids

In [7]:
replaced = p.replace()
h8 = next(iter(replaced))
print(replaced[h8][:1200])

# НАБОР ДАННЫХ ДЛЯ КЕЙСА <e id="dere-gidy">5</e>

Итоговый отчет: «01_Карточка_учета_фактических_затрат_этап_3.pdf».

Проверяемая строка отчета: «3 этап, <e id="cusa-zypu">2026</e> г.».

Договор: <e id="vybi-jiku">24/03/15-01</e>.

Период подтверждающих документов: <e id="nyce-jamo">01.01.2026</e>–<e id="hela-gaqu">31.05.2026</e>.

Остальные файлы в папке являются связанными первичными или расчетными документами. Документы представлены в разных форматах и могут отличаться структурой, расположением реквизитов и формулировками назначения расходов. Наименование статьи КФЗ не обязательно указано в документе дословно, связь необходимо определять по реквизитам и содержанию операции.

При проверке необходимо учитывать номер документа, договор, период, содержание операции и сумму. В наборе могут встречаться повторные документы, документы другого периода или другого договора.

Часть детализированных показателей КФЗ может не иметь отдельного подтверждающего документа. Такие показатели следует от

## Stage 4 — scope checks (contract / period)

In [8]:
scope = p.scope()
print("target label: ", scope.get("target_label"))
print("target row entities:", len(scope.get("target_row") or []),
      "(rest of the card becomes trusted constants)")
print("oos:", {k: v["reasons"] for k, v in scope["oos"].items()})
scope.get("target_row")

target label:  3 этап, 2026 г.
target row entities: 17 (rest of the card becomes trusted constants)
oos: {'b4c8366e': ['contract'], 'f3473dac': ['period']}


['bagu-tudi',
 'foge-kihi',
 'gona-qoba',
 'laxo-fobi',
 'nezy-qewo',
 'piri-qoky',
 'pony-tuhi',
 'popo-nymi',
 'saqu-mofi',
 'sica-rexu',
 'tejy-fitu',
 'wofu-syhu',
 'xeki-hadi',
 'xepa-gixu',
 'xibo-jyro',
 'xoza-kome',
 'zydy-dize']

## Stage 5 — relation extraction (LLM, whole corpus)

In [9]:
# rel_model is already set to the tag model in the config cell above
# (the old "~deepseek/deepseek-pro-latest" id 404s on this account).
# Switch here if you want to A/B another model for relation extraction:
# p.cfg.rel_model = "deepseek/deepseek-v4.1-flash-20261017"

In [10]:
relations_raw = p.relations()
len(relations_raw), relations_raw[:5]

(190,
 [{'equation': 'qeba-faqu == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': '01.01.2026 <= mala-siry && mala-siry <= 31.05.2026',
   'tag': 'period',
   'source': 'scope'},
  {'equation': 'qyqu-niki == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': '01.01.2026 <= loqa-peny && loqa-peny <= 31.05.2026',
   'tag': 'period',
   'source': 'scope'},
  {'equation': 'rari-juqa == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'}])

## Stage 6 — solve the system

In [11]:
rels, fishy, verdicts, dangling = p.solve()   # scope -> only target row stays variable
import pandas as pd
rows = [{"i": r.index, "equation": r.equation, "tag": r.tag,
         "status": r.status, "touches_card": r.touches_card,
         "error": r.error} for r in rels]
df = pd.DataFrame(rows)
print(df["status"].value_counts().to_dict(), "| card-touching:",
      df[df.touches_card]["status"].value_counts().to_dict())
df[df.status != "true"]

{'true': 135, 'false': 53, 'error': 2} | card-touching: {'true': 82, 'false': 48, 'error': 1}


,i,equation,tag,status,touches_card,error
30,30,"raro-geku == ""24/03/15-01""",contract,false,False,string identity comparison is false
41,41,01.01.2026 <= fyxe-popu && fyxe-popu <= 31.05....,period,false,False,contradicts documents or accepted relations
42,42,01.01.2026 <= moso-dyhy && moso-dyhy <= 31.05....,period,false,False,contradicts documents or accepted relations
58,58,sany-xewy == vujy-daca + roni-myve,agg,false,True,contradicts documents or accepted relations
60,60,hino-taje == xany-mebu + hezy-cefe,agg,false,True,contradicts documents or accepted relations
64,64,sany-xewy + xibo-jyro == hino-taje,agg,false,True,contradicts documents or accepted relations
68,68,rasa-vyto == viza-wyki + poxi-nuqa,agg,false,True,contradicts documents or accepted relations
69,69,viza-wyki == fegy-cejo + kogi-xyny,agg,false,True,contradicts documents or accepted relations
70,70,kogi-xyny == ceme-zimy + vino-fedi,agg,false,True,contradicts documents or accepted relations
72,72,sany-xewy == vujy-daca + roni-myve,agg,false,True,contradicts documents or accepted relations


## Stage 7 — report

In [12]:
report = p.report()
from checkier.report import format_report_text
print(format_report_text(report, p._mapping))

=== SUMMARY ===
card cells: 17 total | 15 ok | 1 mismatch | 1 unverified
relations: 190 total | 135 true | 53 false | 0 unverifiable
files: 1 duplicates | 2 out-of-scope | 2 dangling | 0 fishy

=== CARD CELL MISMATCHES (1) ===
  [amount] '1 671 359,96' (id=xibo-jyro, file=card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md)  stated=1671359.96  expected=3104576.46  diff=1433216.50

=== CARD CELLS UNVERIFIED — no equation pinned them down (1) ===
  [number] '0,00' (id=bagu-tudi, file=card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md)  stated=0.00

=== RELATIONS NOT PROVEN TRUE — touching card (49 of 190) ===
  #58 FALSE [agg] sany-xewy == vujy-daca + roni-myve
      where: sany-xewy = '37 272,94' [amount = 37272.94, card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md]; vujy-daca = '37 269,26' [amount = 37269.26, card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md]; roni-myve = '16 163,39' [amount = 16163.39, card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md]
     

In [13]:
import pandas as pd
pd.DataFrame(report["card"]["cells"])

,id,num,mnemonic,raw,type,status,stated,expected,diff,file
0,wofu-syhu,19,wofu-syhu,30,number,ok,30,30,NaN,0dff6363
1,nezy-qewo,20,nezy-qewo,мая 2026,date,ok,739737,739737,NaN,0dff6363
2,foge-kihi,74,foge-kihi,"34,14",number,ok,34.14,34.14,NaN,0dff6363
3,bagu-tudi,75,bagu-tudi,"0,00",number,unverified,0.00,NaN,NaN,0dff6363
4,popo-nymi,76,popo-nymi,"5 558 474,83",amount,ok,5558474.83,5558474.83,NaN,0dff6363
5,tejy-fitu,77,tejy-fitu,"5 248 019,04",amount,ok,5248019.04,5248019.04,NaN,0dff6363
6,saqu-mofi,78,saqu-mofi,"5 054 470,26",amount,ok,5054470.26,5054470.26,NaN,0dff6363
7,xeki-hadi,79,xeki-hadi,"193 548,78",amount,ok,193548.78,193548.78,NaN,0dff6363
8,xepa-gixu,80,xepa-gixu,"310 455,79",amount,ok,310455.79,310455.79,NaN,0dff6363
9,xibo-jyro,81,xibo-jyro,"1 671 359,96",amount,mismatch,1671359.96,3104576.46,1433216.50,0dff6363


In [14]:
# raw JSON summaries if you need them programmatically
import json
print(json.dumps(report["summary"], indent=2, ensure_ascii=False))

{
  "card_cells_total": 17,
  "ok": 15,
  "mismatch": 1,
  "unverified": 1,
  "duplicates": 1,
  "out_of_scope": 2,
  "dangling": 2,
  "fishy": 0,
  "relations_total": 190,
  "relations_true": 135,
  "relations_false": 53,
  "relations_false_on_card": 48,
  "relations_unverifiable": 0
}
